# UN Immigration Data Cleaning Assignment
**Rahat Pervez**

This notebook prepares UN immigration data for Canada from 1980 through 2013.
The final table uses country names as the row index and shows each country's
continent, region, development classification, annual counts, and total.

In [1]:
import pandas as pd
from google.colab import files

# Upload the Excel workbook downloaded from the course link.
uploaded = files.upload()
file_name = next(iter(uploaded))
workbook = pd.ExcelFile(file_name)

print("Workbook tabs:", workbook.sheet_names)

Saving Canada.xlsx to Canada.xlsx
Workbook tabs: ['Regions by Citizenship', 'Canada by Citizenship']


## Load and inspect the data

The workbook may contain title rows above the table. I locate the actual
header row before importing the data.

In [2]:
sheet_name = "Canada by Citizenship"

# Read the first 50 rows without treating any row as the header.
preview = pd.read_excel(workbook, sheet_name=sheet_name, header=None, nrows=50)

# Find the row containing the source column names.
required_headers = {"OdName", "AreaName", "RegName", "DevName"}
header_rows = [
    row_number
    for row_number, row in preview.iterrows()
    if required_headers.issubset(row.dropna().astype(str).str.strip())
]

if not header_rows:
    raise ValueError("Could not find the expected column headers.")

raw = pd.read_excel(workbook, sheet_name=sheet_name, header=header_rows[0])
print("Header found on Excel row", header_rows[0] + 1)
display(raw.head(3))

Header found on Excel row 21


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331


## Clean and reshape the table

I rename the geographic columns, retain all 34 years from 1980 through 2013,
and calculate the total for each country. I then set Country as the row index.

In [3]:
# Excel can load the year headers as numbers. Use consistent string labels.
raw.columns = raw.columns.map(str).str.strip()
years = [str(year) for year in range(1980, 2014)]

renamed = raw.rename(columns={
    "OdName": "Country",
    "AreaName": "Continent",
    "RegName": "Region",
})

# Keep the columns shown in the assignment and remove rows missing
# the geographic labels needed for the country-level table.
columns = ["Country", "Continent", "Region", "DevName", *years]
immigration = renamed[columns].dropna(
    subset=["Country", "Continent", "Region", "DevName"]
).copy()

# Convert the annual counts to numbers and calculate the row total.
immigration[years] = immigration[years].apply(pd.to_numeric)
immigration["Total"] = immigration[years].sum(axis=1, min_count=len(years))

immigration = immigration.set_index("Country").sort_index()
display(immigration.head())

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15


## Validate the results

I compare the first five country totals with the assignment screenshot and
check that the table has all 34 year columns.

In [5]:
expected_totals = {
    "Afghanistan": 58639,
    "Albania": 15699,
    "Algeria": 69439,
    "American Samoa": 6,
    "Andorra": 15,
}

assert immigration.index.name == "Country"
assert immigration.index.is_unique, "Each country should appear once."
assert list(immigration.columns) == ["Continent", "Region", "DevName", *years, "Total"]

for country, expected_total in expected_totals.items():
    actual_total = immigration.loc[country, "Total"]
    assert actual_total == expected_total, f"Total does not match for {country}."

print("Validation passed. The table layout and all five sample totals match.")

Validation passed. The table layout and all five sample totals match.
